# 第16回　ANN と CNN・モデル比較
***
> **前提**: 第15回 SimpleMLP の続きです。第7回の線形回帰に相当する **LinearNet**（隠れ層なし），多層 **DeepMLP**，**SimpleCNN** の3モデルを同条件で学習し精度を比較します。第12回で学んだ混同行列も使います。

## 目次
1. 3モデルの実装
2. 学習と比較
3. 精度の棒グラフ
4. 混同行列とモデル保存

---

## この回で学ぶこと

### なぜ CNN が画像に強いのか

全結合層（MLP）の問題点：
- 28×28 = 784 ピクセルをすべてバラバラに扱う
- 「隣接するピクセルが関連している」という画像の構造的な情報を捨てる
- 画像が少し移動・回転するだけで全く異なる入力になる

CNN（Convolutional Neural Network）はこれを解決する：

```
【畳み込みの直感】
3×3 のフィルター（カーネル）が画像上をスライドしながら
局所的な特徴（エッジ，テクスチャ，形状）を検出する

第1層: エッジ検出（縦線，横線，斜め線）
第2層: テクスチャ（格子，波，点）
第3層: 形状（目，耳，数字の丸みなど）
```

### 畳み込み後の特徴マップサイズの計算

```
出力サイズ = (入力サイズ - カーネルサイズ + 2 × パディング) / ストライド + 1
```

今回の SimpleCNN：
```
入力: (28, 28)
Conv2d(1→16, kernel=3, padding=1): (28+2×1-3)/1+1 = 28 → (28, 28)
MaxPool2d(2): 28 / 2 = 14 → (14, 14)
Conv2d(16→32, kernel=3, padding=1): → (14, 14)
MaxPool2d(2): 14 / 2 = 7 → (7, 7)
→ 32チャネル × 7 × 7 = 1568次元 にFlatten
```

### MaxPooling の役割

- **ダウンサンプリング**：特徴マップを縮小して計算量を削減
- **位置不変性**：特徴が少し移動しても同じ出力（数字「3」が少し右に寄っていても認識できる）
- 各 2×2 領域の最大値を取るため，最も顕著な特徴を保持する

### モデルの保存と読み込み（state_dict）

学習済みモデルを保存する方法：
```python
torch.save(model.state_dict(), "model.pth")  # 重みパラメータのみ保存
```
モデル全体（構造 + 重み）を保存することも可能だが，`state_dict`（重みのみ）の保存が推奨される理由は：
- ファイルサイズが小さい
- Python バージョン間の互換性が高い
- 別のモデルに重みを転用しやすい（転移学習）

In [ ]:
%pip install -q torch torchvision


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

DATA_ROOT = "./data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


## 問題1　3モデルの実装
***

### 3モデルの役割と比較の意義

今回実装する3モデルは，複雑さの段階的な比較になっている：

| モデル | 複雑さ | 特徴 |
|---|---|---|
| `LinearNet` | 最も単純（線形） | 隠れ層なし。784→10 の直接マッピング。ベースライン |
| `DeepMLP` | 中程度（非線形，多層） | 隠れ層2つ。非線形パターンを学習できる |
| `SimpleCNN` | 最も複雑（空間構造を活用） | 畳み込みで画像の局所特徴を抽出 |

同じデータ・同じ条件で比較することで，「モデルの複雑さ」の効果を純粋に検証できる。

### `DeepMLP` の実装について

`SimpleMLP`（問題15）との違いは隠れ層が2つになること：
```
784 → [Linear(784,256)] → [ReLU] → [Linear(256,128)] → [ReLU] → [Linear(128,10)]
```

層を増やすと：
- より複雑なパターンを学習できる
- 計算量が増える
- 過学習しやすくなる（Dropoutなどの正則化が必要になることも）

### `SimpleCNN` の実装について

CNN では `nn.Sequential` を使うと層をまとめて書ける：
```python
self.conv_block = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
)
```

### 課題

以下の3クラスを `nn.Module` として実装してください。

| クラス名 | 構成 | 備考 |
|---|---|---|
| `LinearNet` | 784 → 10 | 第7回線形モデルに相当するベースライン |
| `DeepMLP` | 784 → 256 → 128 → 10（ReLU） | 多層全結合 ANN |
| `SimpleCNN` | Conv(1→16,k=3,p=1)→ReLU→MaxPool(2)→Conv(16→32,k=3,p=1)→ReLU→MaxPool(2)→Flatten→Linear(32×7×7→128)→ReLU→Linear(128→10) | 畳み込み |

実装後，各モデルのパラメータ数を `sum(p.numel() for p in model.parameters())` で確認してください。

#### Hints
- CNN の `forward` では畳み込みブロックを通した後、全結合層に渡す前に flatten が必要。`view(x.size(0), -1)` でバッチサイズを保ったまま平坦化できる
- 全結合層の入力次元は、畳み込みとプーリングを経た後の特徴マップのサイズで決まる。2回の MaxPool2d(2) を経ると 28 → 14 → 7 になる（`32 × 7 × 7 = 1568`）
- 実装後に `print(model)` でパラメータ数を `sum(p.numel() for p in model.parameters())` で確認しよう

In [ ]:
class LinearNet(nn.Module):
    def __init__(self):
        super().__init__()
        # ここにあなたのコードを書いてください

    def forward(self, x):
        # ここにあなたのコードを書いてください
        pass


class DeepMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # ここにあなたのコードを書いてください

    def forward(self, x):
        # ここにあなたのコードを書いてください
        pass


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # ここにあなたのコードを書いてください

    def forward(self, x):
        # ここにあなたのコードを書いてください
        pass


## 問題2　3モデルの学習と比較
***

### 「同条件」で比較することの重要性

科学的な比較には **公平な条件** が必要だ。今回は：
- エポック数：3（全モデル同じ）
- 学習率：0.001（全モデル同じ）
- バッチサイズ：64（全モデル同じ）
- データ：MNIST（全モデル同じ）

これらを統一することで，「モデルのアーキテクチャの違い」だけが結果に影響する。卒業研究でモデルを比較するときも，比較条件を統一することが必須だ。

### `train_one_model` 関数を設計する理由

同じ学習ループを3回書くのは非効率で，ミスも起きやすい。関数化することで：
- コードの重複を排除
- 引数を変えるだけで異なるモデル/設定を試せる
- バグが1箇所に集中する（修正が楽）

### 課題

以下の `train_one_model` 関数を完成させ，3モデルを **同条件**（epoch=3, Adam lr=0.001, batch_size=64）で学習してください。

各モデルのテスト正解率を出力してください。

> **予想**: LinearNet < DeepMLP < SimpleCNN の順で精度が高くなると予想されるが，実際どうなるか確認しよう。CNNはMLPよりどれだけ精度が高いか？

#### Hints
- 関数の構造は「① 学習ループ（q15 と同じ）→ ② テストデータで評価 → ③ 正解率を return」という3段階
- 関数の引数 `model` は `LinearNet()`, `DeepMLP()`, `SimpleCNN()` のいずれかが渡ってくる。どのモデルでも動くように汎用的に書く
- 学習と評価のループは q15 で実装したものを参考にすること
- 3モデルをループで学習する場合、各モデルのインスタンスは新しく生成する必要がある（学習済みの重みを使い回さないため）

In [ ]:
def train_one_model(model, train_loader, test_loader, epochs=3, lr=0.001):
    # ここにあなたのコードを書いてください
    pass


# 3モデルの学習
# ここにあなたのコードを書いてください


## 問題3　精度比較の可視化
***

### 棒グラフで比較する際のポイント

精度の差が小さい（例：97% vs 99%）場合，y軸を 0.95〜1.0 に絞ると差が見やすくなる：

```python
plt.ylim(0.95, 1.0)
```

ただし **y軸を切り捨てる（truncated y-axis）** のは，差を誇張して見せることになるため，学術的な文脈では注意が必要だ。常に「実際の差の大きさ」を意識すること。

### 課題

問題2の結果を**棒グラフ**で可視化し，最高精度のモデル名を出力してください。

棒の上に精度の数値（小数点2桁まで，%表示）を `plt.text` で追加してください。

> **考えてみよう**: モデルの複雑さ（パラメータ数）と精度はどのような関係にあるか？パラメータが多いほど常に精度が高くなるか？


In [ ]:
# 精度比較の棒グラフ
# ここにあなたのコードを書いてください


## 問題4　混同行列の解釈とモデル保存
***

### 混同行列から何を読み取るか

10クラス（数字0〜9）の混同行列は 10×10 のヒートマップになる。対角線が正解，対角線以外が誤分類だ。

よく混同される数字の例：
- **1 と 7**: 縦棒の形が似ている
- **3 と 8**: 右側のカーブが似ている（特に崩した字体）
- **4 と 9**: 縦線と右下のカーブが似ている
- **5 と 6**: どちらも上が開いた形

「どのクラスが最も多く誤分類されているか」を確認し，その数字の特徴を考えてみよう。

### 混同行列から改善策を考える

特定のクラスで誤分類が多い場合の対処法：
- そのクラスのデータを増やす（Data Augmentation）
- クラス不均衡対策（第12回の `class_weight`）
- モデルを複雑にする
- 前処理の改善（画像の回転・スケール正規化）

### モデルの保存

第17回では保存した `best_mnist_model.pth` を読み込んで使用する。**ファイル名と保存先を正確に一致させること**が重要だ。

### 課題

最高精度モデルでテストデータ全体の予測を行い，**混同行列**を seaborn ヒートマップで可視化してください。

ヒートマップには軸ラベル（予測/正解）とタイトルを付けてください。

混同行列を見て，「どの数字同士が混同されやすいか」を print で2〜3個コメントしてください。

さらに，最高精度モデルの `state_dict` を `best_mnist_model.pth` として保存してください（第17回で使用）。

#### Hints
- テストデータ全体の予測を集めるには、バッチごとにリストに追加していく。`list.extend()` が使いやすい
- GPU 上の Tensor を NumPy に変換するには、先に `.cpu()` でCPUに移してから `.numpy()` を呼ぶ
- `confusion_matrix(y_true, y_pred)` で10×10の行列が得られる。`sns.heatmap` に `annot=True, fmt="d"` を渡すと数値が表示される
- モデルの保存は `torch.save(model.state_dict(), "ファイル名.pth")` で行う。第17回でこのファイルを使う

In [ ]:
# 混同行列とモデル保存
# ここにあなたのコードを書いてください
